<a href="https://colab.research.google.com/github/ahmedalmnaweer-ui/Smart-HSA-HQ-/blob/main/Copy_of_Untitled7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

%%writefile app.py
import streamlit as st
import pandas as pd
import plotly.express as px
from datetime import datetime

# إعدادات الصفحة يجب أن تكون أول سطر
st.set_page_config(page_title="HSA Master ERP", page_icon="🏢", layout="wide")

# تصميم CSS
st.markdown('''
    <style>
    .stApp { direction: rtl; font-family: 'Tajawal', sans-serif; background-color: #F4F6F9;}
    .card { background-color: white; padding: 20px; border-radius: 12px; box-shadow: 0 4px 10px rgba(0,0,0,0.05); border-right: 5px solid #2980b9; margin-bottom: 20px;}
    .total-box { background-color: #27ae60; color: white; padding: 20px; border-radius: 10px; text-align: center; font-size: 26px; font-weight: bold;}
    .header-box { background-color: #1A5276; color: white; padding: 15px; border-radius: 8px; text-align: center;}
    </style>
''', unsafe_allow_html=True)

st.markdown("<div class='header-box'><h1>🏢 نظام HSA لإدارة المشاريع والتحليل الهندسي</h1><p>حصر | تسعير | جداول كميات | إنتاجية | مخططات</p></div><br>", unsafe_allow_html=True)

# تهيئة الجلسات (Session State)
if 'currency_usd' not in st.session_state: st.session_state.currency_usd = 530.0
if 'currency_sar' not in st.session_state: st.session_state.currency_sar = 140.0
if 'boq_data' not in st.session_state: st.session_state.boq_data = []
if 'volume' not in st.session_state: st.session_state.volume = 1.0

# تحميل البيانات الافتراضية
@st.cache_data
def load_default_data():
    materials = pd.DataFrame({
        "الصنف": ["أسمنت بورتلاندي", "حديد تسليح 14مم", "بلك أسمنتي 20", "كوع زاوية 6 انش", "سلك 16 ملم الفنار", "مغسلة إفرنجي"],
        "الوحدة": ["كيس", "طن", "حبة", "حبة", "لفة", "طقم"],
        "سعر السوق (ريال)": [4500, 520000, 250, 1500, 15000, 113000]
    })
    labor = pd.DataFrame({
        "المهنة": ["معلم نجار", "معلم حداد", "معلم بناء", "معلم سباك", "عامل عادي"],
        "الأجر اليومي (ريال)": [15000, 18000, 15000, 20000, 6000],
        "الإنتاجية اليومية": [2.5, 0.250, 44.0, 10.0, 1.0],
        "وحدة الإنتاجية": ["م3", "طن", "م2", "نقطة", "مقطوعية"]
    })
    equip = pd.DataFrame({
        "الآلية": ["حفار بوكلين", "خلاطة مركزية", "ونش رفع", "هزاز خرسانة", "بدون آليات"],
        "التكلفة اليومية (ريال)": [150000, 200000, 40000, 5000, 0],
        "الإنتاجية اليومية": [300, 100, 1.0, 100, 1.0],
        "وحدة الإنتاجية": ["م3", "م3", "مقطوعية", "م3", "مقطوعية"]
    })
    return materials, labor, equip

df_mat, df_lab, df_eq = load_default_data()

# القائمة الجانبية (العملات)
with st.sidebar:
    st.title("💱 لوحة العملات")
    st.session_state.currency_usd = st.number_input("سعر صرف الدولار ($):", value=st.session_state.currency_usd)
    st.session_state.currency_sar = st.number_input("سعر صرف السعودي (SAR):", value=st.session_state.currency_sar)
    st.markdown("---")

tabs = st.tabs(["📚 قاعدة البيانات", "📐 الحصر الهندسي", "⚙️ تحليل الأسعار", "📋 جدول الكميات", "📈 لوحة القيادة"])

# التبويب الأول
with tabs[0]:
    st.markdown("<div class='card'><h3>📂 بيانات الأسعار والإنتاجية</h3></div>", unsafe_allow_html=True)
    col1, col2, col3 = st.columns(3)
    with col1:
        st.write("**أسعار المواد:**")
        st.dataframe(df_mat, height=250, use_container_width=True)
    with col2:
        st.write("**أجور وإنتاجية العمالة:**")
        st.dataframe(df_lab, height=250, use_container_width=True)
    with col3:
        st.write("**تكاليف وإنتاجية المعدات:**")
        st.dataframe(df_eq, height=250, use_container_width=True)

# التبويب الثاني
with tabs[1]:
    st.markdown("<div class='card'><h3>📐 حساب حصر الخرسانة وتفريد الحديد</h3></div>", unsafe_allow_html=True)
    c1, c2 = st.columns(2)
    with c1:
        elem_type = st.selectbox("نوع العنصر الإنشائي:", ["قواعد مسلحة", "أعمدة", "ميد (جسور أرضية)", "بلاطات أسقف"])
        l = st.number_input("الطول (متر):", value=2.0, min_value=0.0)
        w = st.number_input("العرض (متر):", value=2.0, min_value=0.0)
        d = st.number_input("السمك (متر):", value=0.5, min_value=0.0)
        n = st.number_input("العدد:", value=10, min_value=1)
    with c2:
        steel_density = st.number_input("كثافة الحديد (كجم/م3):", value=110.0, min_value=0.0)
        waste_ratio = st.slider("نسبة الهالك %:", 0.0, 15.0, 5.0)

        if st.button("حساب الكميات واعتمادها", use_container_width=True):
            vol = l * w * d * n
            vol_waste = vol * (1 + (waste_ratio/100))
            steel_ton = ((vol * steel_density) * (1 + (waste_ratio/100))) / 1000
            st.session_state.volume = vol_waste
            st.success(f"**حجم الخرسانة الصافي:** {vol:,.2f} م3 | **مع الهالك:** {vol_waste:,.2f} م3")
            st.error(f"**وزن الحديد المطلوب:** {steel_ton:,.3f} طن")

# التبويب الثالث
with tabs[2]:
    st.markdown("<div class='card'><h3>⚙️ تحليل تكلفة البند (حسب الإنتاجية)</h3></div>", unsafe_allow_html=True)
    item_desc = st.text_input("وصف البند ومواصفاته:", "توريد وتنفيذ قواعد خرسانية مسلحة...")
    item_unit = st.selectbox("وحدة القياس:", ["م3", "م2", "م.ط", "طن", "عدد"])

    p1, p2, p3 = st.columns(3)
    with p1:
        sel_mat = st.selectbox("المادة الأساسية:", df_mat['الصنف'])
        mat_price = df_mat.loc[df_mat['الصنف'] == sel_mat, 'سعر السوق (ريال)'].values[0]
        mat_qty = st.number_input("كمية المادة المطلوبة لـ 1 وحدة:", value=1.0, min_value=0.0)
        total_mat = mat_price * mat_qty
        st.info(f"التكلفة: {total_mat:,.2f} ريال")

    with p2:
        sel_lab = st.selectbox("العمالة المنفذة:", df_lab['المهنة'])
        lab_wage = df_lab.loc[df_lab['المهنة'] == sel_lab, 'الأجر اليومي (ريال)'].values[0]
        lab_prod = df_lab.loc[df_lab['المهنة'] == sel_lab, 'الإنتاجية اليومية'].values[0]
        labor_cost = lab_wage / lab_prod if lab_prod > 0 else 0
        st.warning(f"التكلفة: {labor_cost:,.2f} ريال")

    with p3:
        sel_eq = st.selectbox("الآليات والمعدات:", df_eq['الآلية'])
        eq_wage = df_eq.loc[df_eq['الآلية'] == sel_eq, 'التكلفة اليومية (ريال)'].values[0]
        eq_prod = df_eq.loc[df_eq['الآلية'] == sel_eq, 'الإنتاجية اليومية'].values[0]
        eq_cost = eq_wage / eq_prod if eq_prod > 0 else 0
        st.success(f"التكلفة: {eq_cost:,.2f} ريال")

    m1, m2 = st.columns(2)
    sup_margin = m1.number_input("ربح التوريد %:", value=10.0, min_value=0.0)
    exe_margin = m2.number_input("ربح التنفيذ والإداريات %:", value=15.0, min_value=0.0)

    direct_cost = total_mat + labor_cost + eq_cost
    sup_profit = total_mat * (sup_margin/100)
    exe_profit = direct_cost * (exe_margin/100)
    final_price = direct_cost + sup_profit + exe_profit

    st.markdown(f'''<div style="background-color: #E8F8F5; padding: 15px; border-radius: 8px; border: 1px dashed #1ABC9C;">
        <h3 style="color:#16A085;">سعر الوحدة النهائي: {final_price:,.2f} ريال يمني</h3>
        </div><br>''', unsafe_allow_html=True)

    if st.button("📥 إضافة البند إلى المقايسة (BOQ)", type="primary", use_container_width=True):
        st.session_state.boq_data.append({
            "وصف البند ومواصفاته": item_desc,
            "الوحدة": item_unit,
            "الكمية": st.session_state.volume,
            "سعر الوحدة (ريال)": final_price,
            "تكلفة المواد": total_mat,
            "تكلفة العمالة والمعدات": labor_cost + eq_cost
        })
        st.success("✅ تمت الإضافة لجدول الكميات بنجاح! الكمية تم جلبها من شاشة الحصر تلقائياً.")

# التبويب الرابع
with tabs[3]:
    st.markdown("<div class='card'><h3>📋 جدول بنود الدراسة وتكلفة المشروع (BOQ)</h3></div>", unsafe_allow_html=True)
    if st.session_state.boq_data:
        df_boq = pd.DataFrame(st.session_state.boq_data)

        edited_df = st.data_editor(
            df_boq,
            column_config={
                "الكمية": st.column_config.NumberColumn("الكمية", required=True, min_value=0.0),
                "سعر الوحدة (ريال)": st.column_config.NumberColumn(disabled=True),
                "وصف البند ومواصفاته": st.column_config.TextColumn(disabled=True)
            },
            num_rows="dynamic", use_container_width=True
        )

        st.session_state.boq_data = edited_df.to_dict('records')
        edited_df['الإجمالي (ريال)'] = edited_df['الكمية'] * edited_df['سعر الوحدة (ريال)']

        grand_total_yer = edited_df['الإجمالي (ريال)'].sum()
        grand_total_usd = grand_total_yer / st.session_state.currency_usd
        grand_total_sar = grand_total_yer / st.session_state.currency_sar

        st.markdown(f'''<div class="total-box">
            الإجمالي الكلي: {grand_total_yer:,.2f} ريال يمني <br>
            <span style="font-size: 20px; color: #F1C40F;">يعادل: {grand_total_usd:,.2f} $ | {grand_total_sar:,.2f} SAR</span>
            </div>''', unsafe_allow_html=True)
    else:
        st.info("المقايسة فارغة. قم بتحليل وإضافة بنود من تبويب 'تحليل الأسعار'.")

# التبويب الخامس
with tabs[4]:
    st.markdown("<div class='card'><h3>📈 المخططات البيانية لتحليل المشروع</h3></div>", unsafe_allow_html=True)
    if st.session_state.boq_data:
        df_dash = pd.DataFrame(st.session_state.boq_data)
        df_dash['إجمالي التكلفة'] = df_dash['الكمية'] * df_dash['سعر الوحدة (ريال)']
        df_valid = df_dash[df_dash['الكمية'] > 0]

        if not df_valid.empty:
            d1, d2 = st.columns(2)
            with d1:
                st.write("**توزيع الميزانية على البنود:**")
                fig_pie = px.pie(df_valid, values='إجمالي التكلفة', names='وصف البند', hole=0.4)
                st.plotly_chart(fig_pie, use_container_width=True)
            with d2:
                st.write("**المواد مقابل العمالة والمعدات:**")
                tot_mat = (df_valid['تكلفة المواد'] * df_valid['الكمية']).sum()
                tot_lab_eq = (df_valid['تكلفة العمالة والمعدات'] * df_valid['الكمية']).sum()
                fig_bar = px.bar(x=['المواد والتوريدات', 'العمالة والمعدات'], y=[tot_mat, tot_lab_eq], color=['المواد والتوريدات', 'العمالة والمعدات'], labels={'x':'النوع', 'y':'التكلفة'})
                st.plotly_chart(fig_bar, use_container_width=True)
        else:
            st.warning("أدخل الكميات في المقايسة لظهور المخططات.")
    else:
        st.info("لا توجد بيانات لعرض المخططات.")

In [ ]:

!pip install streamlit pandas plotly openpyxl -q
import urllib.request
import time

# استخراج كلمة المرور (IP Address)
print("=====================================================")
print("🔑 كلمة المرور الخاصة بك هي:")
ip_address = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n")
print(f"-> {ip_address} <-")
print("👉 انسخ هذا الرقم لأن الموقع سيطلبه منك عند فتح الرابط 👈")
print("=====================================================")

# إيقاف أي عمليات Streamlit سابقة (لتجنب تعليق المنافذ)
!pkill -f streamlit

# تشغيل التطبيق في الخلفية
!nohup streamlit run app.py &

# الانتظار لثانيتين لضمان عمل التطبيق
time.sleep(2)

# فتح المنفذ مع الموافقة التلقائية (-y) لتجاوز رسالة التأكيد
!npx -y localtunnel --port 8501